<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://github.com/statmike/scale-forecasting/blob/main/notebooks/05_spark_naive.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo">
      <br>View on<br>GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F05_spark_naive.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/statmike/scale-forecasting/blob/main/notebooks/05_spark_naive.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Colab logo">
      <br>Run in<br>Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/statmike/scale-forecasting/blob/main/notebooks/05_spark_naive.ipynb">
      <img width="32px" src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery logo">
      <br>Open in<br>BigQuery Studio
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 05 · Spark **naive** — the straggler anti-pattern

The deliberate counter-example. `spark_method='naive'` buckets on **`ts_id` alone**, so *all* of a series' models land in one Spark task and run **sequentially**. One slow model (`sarimax`) then holds an executor for the series' whole model set while fast cells elsewhere finish and idle — the straggler the `explode` method exists to avoid (notebook 01). Same code, same input, same registry — only the bucket key differs (`bucket_key_cols`, DESIGN §2.1).

Run this to *see* the anti-pattern in the numbers, then compare `run_summary`/`fit_seconds` against the explode run on the same 100 series.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable.

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses (G1), so a notebook run and a Composer run land in the same registry. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. This demo targets the live `run_registry` + `v_model_leaderboard` / `v_run_summary`.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model (with `median_fit_seconds` — the per-cell fit time, the straggler signal here), and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=15, pause=4.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae, median_fit_seconds "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Parameters — edit me

Everything that shapes the run is set here as plain Python, then assembled into a **`RunConfig`** — the one frozen object that drives the run and is logged verbatim to `run_registry.raw_config`, so *this cell is the experiment record* (G2/G3).

- **`SPARK_METHOD='naive'`** is the whole point: bucket on `ts_id`, run a series' models sequentially in one task. Everything else matches the explode notebook so it's an apples-to-apples comparison.
- **`MODELS`** deliberately mixes a slow model (`sarimax`) with fast ones so the straggler shows up in `mean_fit_seconds`.
- **`SERIES_LIMIT=100`** — the C6 demo scale (the *same* first 100 series every approach uses, DESIGN §13.1).

In [ ]:
from scale_forecasting import main
from scale_forecasting.config import RunConfig
from scale_forecasting.registry.ids import make_run_id

# === Parameters — edit me ===============================================
RUN_NAME = f"nb05 spark naive {int(time.time())}"  # timestamped → a fresh run each execution
SOURCE_TABLE = "source_series_iceberg"  # shipped seed (100k); _iceberg↔_native to compare storage
MODELS = ["theta", "holtwinters", "sarimax"]  # sarimax is the straggler that holds the task
SPARK_METHOD = "naive"  # bucket on ts_id — models run sequentially per series (anti-pattern)
HORIZON = 28
SERIES_LIMIT = 100  # the C6 demo scale
HOLIDAYS = ["US"]
# ========================================================================

cfg = RunConfig(
    run_name=RUN_NAME,
    python_runtime="spark",
    spark_method=SPARK_METHOD,
    data={"source_table": SOURCE_TABLE, "horizon": HORIZON, "series_limit": SERIES_LIMIT},
    models=MODELS,
    features={"holidays": HOLIDAYS},
)
run_id = make_run_id(cfg)
print("run_id:", run_id, "| models:", cfg.models, "| method:", cfg.spark_method)

## Run — naive fan-out as a Dataproc batch

`main.run(cfg)` with no injected session submits the identical engine as a remote Dataproc Serverless batch (the proven production path). Because `spark_method='naive'`, the engine buckets on `ts_id` and loops the models per series — the anti-pattern, on-cluster. It returns the same `run_id` we computed above.

In [ ]:
returned = main.run(cfg)
assert returned == run_id
print("naive run complete:", run_id)

## Review — the straggler in the numbers

All three models ran as Spark cells under one `run_id`. Watch `median_fit_seconds`: the slow model dominates the per-series task wall-clock because naive runs them in series. Compare this `run_summary` (and the batch's runtime in the Dataproc console) against the explode run on the *same* 100 series — explode spreads the slow cells across buckets so the run finishes in roughly the slowest *single cell*, not the slowest *series*.

In [ ]:
board = leaderboard(run_id, expect_models=cfg.models)
board

In [ ]:
run_summary(run_id)